# Simple ResNet Model

## Importing Packages

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchinfo
from torchvision import transforms
from torchvision.datasets import ImageFolder

from torch.utils.data import Dataset, DataLoader, random_split

import helper_utils

# Set seed
SEED = 42

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
dataset_path = "data"

In [5]:
# Display the image count statistics for each class.
helper_utils.display_dataset_stats(dataset_path)

Class Name,Number of Images
Agriculture,800
Airport,800
Beach,800
City,800
Desert,800
Forest,800
Grassland,800
Highway,800
Lake,800
Mountain,800


In [6]:
# Pre-calculated mean and std of this dataset
mean = [0.378, 0.393, 0.345]
std = [0.205, 0.173, 0.170]

# Transforms for Training data with augmentations
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(size = (100, 100), scale = (0.8, 1.0)),
    transforms.RandomHorizontalFlip(p = 0.5),
    transforms.RandomVerticalFlip(p = 0.5),
    transforms.RandomRotation(degrees = 15),
    transforms.ColorJitter(brightness = 0.2, contrast = 0.2, saturation = 0.2, hue = 0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean = mean, std = std)
])

# Transforms for Validation and Test data (no augmentations)
val_transforms = transforms.Compose([
    transforms.Resize((100, 100)),
    transforms.ToTensor(),
    transforms.Normalize(mean = mean, std = std)
])

In [10]:
skyview_dataset = ImageFolder(root = dataset_path, transform = None)

# Calculate the number of samples for training, validation and testing
total_samples = len(skyview_dataset)

train_size = int (0.7 * total_samples)
val_size = int(0.15 * total_samples)
test_size = total_samples - train_size - val_size

# Split the dataset into training, validation and testing sets
train_data, val_data, test_data = random_split(skyview_dataset, [train_size, val_size, test_size], generator = torch.Generator().manual_seed(SEED))

# Dataset Class

In [18]:
class ImgDataset(Dataset):
    def __init__ (self, data, transform = None):
        super().__init__()

        self.data = data
        self.transform = transform

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        img, label = self.data[index]

        if self.transform:
            img = self.transform(img)

        return img, label

In [19]:
train_dataset = ImgDataset(data = train_data, transform = train_transforms)
val_dataset = ImgDataset(data = val_data, transform = val_transforms)
test_dataset = ImgDataset(data = test_data, transform = val_transforms)

## DataLoaders

In [20]:
BATCH_SIZE = 32

train_loader = DataLoader(dataset = train_dataset, batch_size = BATCH_SIZE, shuffle = True)
val_loader = DataLoader(dataset = val_dataset, batch_size = BATCH_SIZE, shuffle = False)
test_loader = DataLoader(dataset = test_dataset, batch_size= 1, shuffle = False)

# Simple ResNet Components

## Residual Block

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride = 1, downsample = None):
        # Initialize the parent nn.Module class
        super(ResidualBlock, self).__init__()

        # Fist component of the main path
        self.convblock1 = nn.Sequential(
            nn.Conv2d(
                in_channels = in_channels,
                out_channels = out_channels,
                kernel_size = 3,
                stride = stride,
                padding = 1,
                bias = False
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace = True)
        )

        # Second component of the main path
        self.convblock2 = nn.Sequential(
            nn.Conv2d(
                in_channels = in_channels,
                out_channels = out_channels,
                kernel_size = 3,
                stride = stride,
                padding = 1,
                bias = False
            ),
            nn.BatchNorm2d(out_channels),

        )

        # Optional downsampling layer for the skip connection
        self.downsample = downsample

    def _initial_forward(self, x):
        out = self.convblock1(x)
        out = self.convblock2(out)

        return out

    def forward(self, x):

        # stores the input for the skip connection
        identity = x

        out = self._initial_forward(x)

        # Apply downsampling to the identity if needed to match out dimensions
        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity

        # Apply final ReLU activation after adding the skip connection
        out = F.relu(out)

        return out


## Simple ResNet Class

In [ ]:
class SimpleResNet(nn.Module):
    def __init__(self, num_classes = 5, num_blocks = [2, 2, 2]):
        super(SimpleResNet, self).__init__()

        self.num_classes = num_classes
        self.num_blocks = num_blocks

        self.in_channels = 32

        self.initial_block = self._get_initial_block()

        


    def _make_residual_block(self, out_channels, num_blocks, stride):

        downsample = None

        if stride != 1 or self.in_Channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, 
                          out_channels, 
                          kernel_size = 1,
                          stride = stride,
                          bias = False),
                nn.BatchNorm2d(out_channels)
            )

        layers = []

        first_block = ResidualBlock(in_channels = self.in_channels, out_channels = out_channels, stride = stride, downsample = downsample)

        layers.append(first_block)

        self.in_channels = out_channels

        for _ in range(1, num_blocks):
            layers.append(ResidualBlock(in_channels = out_channels, out_channels = out_channels))

            return nn.Sequential(*layers)
        
        